In [ ]:
# ============================================================================
# PARAMETERS — this cell is identical in all three notebooks.
# ============================================================================
NOTEBOOK_NAME = "02_train_all"    # identity + build fingerprint of THESE cells;
NOTEBOOK_BUILD = "1f282153c5d6"  # checked against the repo so stale cells fail loudly
RUN_MODE = "micro"          # "smoke" | "micro" (default) | "budget" | "full"
NUM_GPUS = None             # None = use every visible GPU; set 1 to force single-GPU
PUBLISH_KAGGLE_DATASET = True
CKPT_DATASET_SLUG = "dentex-repro-ckpts"
DATA_DATASET_SLUG = "dentex-repro-data"
REPO_URL = "https://github.com/christopherh-88/HierarchicalDet.git"

import os, subprocess, sys

# On Kaggle the repo is cloned into /kaggle/working (the only writable place
# that survives "Save Version"); locally the notebook already sits inside it.
if os.path.isdir("/kaggle/working"):
    CLONE = "/kaggle/working/repo"
    if os.path.isdir(os.path.join(CLONE, ".git")):
        subprocess.run(["git", "-C", CLONE, "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "50", REPO_URL, CLONE], check=True)
    PROJECT_ROOT = os.path.join(CLONE, "dentex-repro")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["RUN_MODE"] = RUN_MODE
print("project root:", PROJECT_ROOT)


In [ ]:
# ---- Environment: install, pin, and prove the VENDORED code is what loaded ----
# Kaggle reverts to its base image every session, so this runs every time.
import json
from src import setup_env

# `git pull` above refreshed src/ and configs_repro/ -- but NOT these cells,
# which are the copy uploaded to Kaggle. Fail loudly rather than run a mix.
print("notebook build:", setup_env.assert_notebook_current(NOTEBOOK_NAME, NOTEBOOK_BUILD))
setup_env.install_dependencies()
# The vendored pycocotools ships Python sources only; its compiled `_mask`
# extension is grafted in here and VERIFIED BY IMPORT. It is compiled against
# numpy's C ABI, so a mismatch surfaces as "numpy.dtype size changed" deep
# inside detectron2.structures — which reads as a detectron2 problem and is not.
import numpy
print("numpy {} | pycocotools _mask -> {}".format(
    numpy.__version__, setup_env.ensure_pycocotools_mask()))
run = setup_env.bootstrap(RUN_MODE, require_gpu=True)

from src import manifest, train_utils

NUM_GPUS = NUM_GPUS if NUM_GPUS is not None else max(1, train_utils.visible_gpus())
lock = setup_env.write_requirements_lock()
environment = setup_env.env_report()
manifest.record_environment(environment)

# The repo vendors MODIFIED detectron2 / pycocotools (multi-label partial
# annotations, 3-tier category schema). A pip-installed copy silently shadows
# them and every number changes, so this is an assertion, not a warning.
found = setup_env.assert_vendored()
for module, path in found.items():
    print("{:14s} -> {}".format(module, path))
import detectron2, pycocotools, evaluator                              # noqa: F401
from hierarchialdet.util.coco_3class_eval import COCOEvaluator         # noqa: F401
from hierarchialdet.dataset_mapper_patched import DiffusionDetDatasetMapper  # noqa: F401
print("full import chain OK | commit {} | {} GPU process(es)".format(
    environment["repo_commit"][:12], NUM_GPUS))


In [ ]:
# ---- Inputs, weights, configs ----
from src import data_convert, eval_utils, registration

paths = data_convert.layout()
for key in ("train_quadrant", "train_enumeration", "train_diagnosis", "val_diagnosis",
            "test_diagnosis"):
    assert os.path.exists(paths[key]), (
        "{} is missing — run notebook 01, or attach the {} dataset"
        .format(paths[key], DATA_DATASET_SLUG))

IMAGENET_WEIGHTS = train_utils.ensure_swin_weights()
CFG = {name: os.path.join(setup_env.CONFIGS_REPRO, "diffdet.dentex.{}.yaml".format(name))
       for name in ("quadrant", "enumeration", "diagnosis", "base_diffusiondet")}
NOISY_DIR = os.path.join(setup_env.RUNS_DIR, "noisy_boxes")
os.makedirs(NOISY_DIR, exist_ok=True)
print("ImageNet Swin-B:", IMAGENET_WEIGHTS)
print("GPU-hours already recorded on disk: {:.2f}".format(setup_env.gpu_hours_spent()))


In [ ]:
# ---- Does multi-GPU actually work in this Kaggle session? ----
# Answered by running a real 20-iteration job, not assumed. A failure falls back
# to one GPU and is recorded, rather than being fought for the rest of the study.
#
# The probe uses the QUADRANT stage deliberately: it is the worst case for DDP,
# because its data leaves the enumeration/disease heads unsupervised, so those
# parameters receive no gradient. A probe on the diagnosis stage (all heads
# supervised) once passed while the real quadrant run then died at iteration 2
# -- the probe must exercise the shape that actually breaks.
import shutil

ddp = {"requested": NUM_GPUS, "works": None, "error": None}
probe_dir = os.path.join(setup_env.RUNS_DIR, "ddp_probe")
if NUM_GPUS > 1 and not os.path.exists(os.path.join(setup_env.RUNS_DIR, ".ddp_ok")):
    try:
        train_utils.launch_training(
            CFG["quadrant"],
            train_utils.base_overrides(run, probe_dir, 20, IMAGENET_WEIGHTS, NUM_GPUS),
            registration.training_env("quadrant_train"), probe_dir, NUM_GPUS,
            resume=False, log_name="ddp_probe.log")
        ddp["works"] = True
        open(os.path.join(setup_env.RUNS_DIR, ".ddp_ok"), "w").close()
    except RuntimeError as error:
        ddp["works"] = False
        ddp["error"] = str(error)[-1500:]
        NUM_GPUS = 1
        setup_env.log_deviation(
            "multi-GPU training disabled (DDP failed in this Kaggle session)",
            "launch(num_gpus=2) failed during the probe; the study runs single-GPU "
            "rather than fighting a flaky DDP setup", "02_train_all",
            impact="effective batch size halves relative to a 2-GPU run")
else:
    ddp["works"] = "already verified" if NUM_GPUS > 1 else "single GPU"
shutil.rmtree(probe_dir, ignore_errors=True)
print(json.dumps(ddp, indent=2), "-> NUM_GPUS =", NUM_GPUS)


In [ ]:
# ---- Pre-flight: a short real run + a real evaluation, on 10 images ----
# Proves the whole chain end to end before committing hours of quota to it.
# Skipped when RUN_MODE="smoke", where the real runs below already are exactly
# this. 30 iterations, not 200: at the measured 9.13 s/iter on T4 x2, 200 would
# spend half an hour of the training budget proving plumbing.
PREFLIGHT_ITERS = 30
smoke = {"skipped": run.is_smoke}
if not run.is_smoke and not train_utils.is_complete("preflight"):
    smoke_dir = train_utils.run_dir("preflight")
    train_utils.launch_training(
        CFG["diagnosis"],
        train_utils.base_overrides(run, smoke_dir, PREFLIGHT_ITERS,
                                   IMAGENET_WEIGHTS, NUM_GPUS),
        registration.training_env("diagnosis_train"), smoke_dir, NUM_GPUS, resume=False)
    smoke_weights = os.path.join(smoke_dir, "model_final.pth")
    assert os.path.exists(smoke_weights), "pre-flight produced no model_final.pth"
    smoke_eval = eval_utils.evaluate_checkpoint(
        smoke_weights, CFG["diagnosis"], split="diagnosis_test", tier=2, seed=0, limit=10)
    # 200 iterations cannot detect anything useful; the assertion is that the
    # metric pipeline RAN, not that it scored well.
    assert set(smoke_eval["tiers"]) == {"quadrant", "enumeration", "diagnosis"}
    smoke = {"weights": smoke_weights,
             "metrics": {t: p["metrics"] for t, p in smoke_eval["tiers"].items()}}
    shutil.rmtree(smoke_dir, ignore_errors=True)
print(json.dumps(smoke, indent=2, default=str))


In [ ]:
# ---- Throughput calibration (hour-capped modes only) ----
# 500 real training iterations, 100 warm-up discarded. Cached, so it is paid
# once for the whole study and charged to the budget once, not once per stage.
calibration = None
if run.quadrant.max_iter is None or run.diagnosis.max_iter is None:
    calibration = train_utils.calibrate_rate(
        "swinb_gpu{}".format(NUM_GPUS), CFG["quadrant"], run,
        "quadrant_train", IMAGENET_WEIGHTS, NUM_GPUS)
    print(json.dumps(calibration, indent=2))


In [ ]:
# ---- Stage 0: quadrant ----
training_records = []
quadrant_record = train_utils.train_stage(
    "quadrant_stage", CFG["quadrant"], run, "quadrant_train",
    IMAGENET_WEIGHTS, NUM_GPUS, run.quadrant, calibration=calibration)
quadrant_record["kind"] = "prerequisite"
training_records.append(quadrant_record)
QUADRANT_WEIGHTS = train_utils.final_weights("quadrant_stage")
print(json.dumps({k: v for k, v in quadrant_record.items() if k != "launch"},
                 indent=2, default=str)[:2000])


In [ ]:
# ---- Quadrant model -> noisy boxes for the enumeration stage ----
# This is the manipulation signal: tier k-1's detections over tier k's own train
# and validation images, filtered at score >= 0.5 by the dataset mapper.
enum_boxes = {}
for key, split in (("NOISY_BOX_TRAIN", "quadrant_enumeration_train"),
                   ("NOISY_BOX_VAL", "diagnosis_val")):
    output = os.path.join(NOISY_DIR, "quadrant_over_{}.json".format(split))
    if not os.path.exists(output):
        print(json.dumps(eval_utils.dump_predictions(
            QUADRANT_WEIGHTS, CFG["quadrant"], split, 0, output, seed=0,
            limit=run.eval_limit), indent=2)[:900])
    enum_boxes[key] = output
print(enum_boxes)


In [ ]:
# ---- Stage 1: enumeration (transfer + manipulation from stage 0) ----
enumeration_record = train_utils.train_stage(
    "enumeration_stage", CFG["enumeration"], run, "quadrant_enumeration_train",
    QUADRANT_WEIGHTS, NUM_GPUS, run.enumeration, calibration=calibration,
    noisy_boxes=enum_boxes)
enumeration_record["kind"] = "prerequisite"
training_records.append(enumeration_record)
ENUM_WEIGHTS = train_utils.final_weights("enumeration_stage")
print(json.dumps({k: v for k, v in enumeration_record.items() if k != "launch"},
                 indent=2, default=str)[:2000])


In [ ]:
# ---- Enumeration model -> noisy boxes for the diagnosis stage ----
from src import degradations

diagnosis_boxes = {}
for key, split in (("NOISY_BOX_TRAIN", "diagnosis_train"),
                   ("NOISY_BOX_VAL", "diagnosis_val")):
    output = os.path.join(NOISY_DIR, "enumeration_over_{}.json".format(split))
    if not os.path.exists(output):
        print(json.dumps(eval_utils.dump_predictions(
            ENUM_WEIGHTS, CFG["enumeration"], split, 1, output, seed=0,
            limit=run.eval_limit), indent=2)[:900])
    diagnosis_boxes[key] = output

# Also produced here because notebook 03's fault-injection experiment needs the
# prior tier's detections over the *test* split.
prior_test = os.path.join(NOISY_DIR, "enumeration_over_diagnosis_test.json")
if not os.path.exists(prior_test):
    eval_utils.dump_predictions(ENUM_WEIGHTS, CFG["enumeration"], "diagnosis_test", 1,
                                prior_test, seed=0, limit=run.eval_limit)

box_stats = {key: degradations.summarize_prediction_file(path)
             for key, path in diagnosis_boxes.items()}
print(json.dumps(box_stats, indent=2))


In [ ]:
# ---- Train every variant ----
plans = train_utils.variant_plan(run, {"enumeration": ENUM_WEIGHTS},
                                 IMAGENET_WEIGHTS, diagnosis_boxes)
variant_records = []
for plan in plans:
    print("\n" + "=" * 72)
    print("{}  (transfer={}, manipulation={})".format(
        plan["variant"], plan["switches"]["transfer"], plan["switches"]["manipulation"]))
    print("=" * 72)
    record = train_utils.train_stage(
        plan["run_name"], CFG["diagnosis"], run, "diagnosis_train",
        plan["weights"], NUM_GPUS, run.diagnosis, calibration=calibration,
        noisy_boxes=plan["noisy_boxes"],
        trajectory_fractions=run.trajectory_fractions)
    record.update({"variant": plan["variant"], "label": plan["label"],
                   "switches": plan["switches"], "kind": "diagnosis_variant"})
    variant_records.append(record)
    training_records.append(record)
    print("iterations {} | wall {} s | stopped on time budget: {}".format(
        record.get("max_iter"), record.get("wall_seconds"),
        record.get("stopped_on_time_budget")))

# Different wall time between variants is fine (manipulation costs extra
# dataloading). Different iteration counts, seeds or batch sizes are not — that
# would make the comparison measure the budget instead of the switch.
matched = train_utils.assert_matched_budgets(variant_records)
print("\n" + json.dumps(matched, indent=2, default=str))


In [ ]:
# ---- Base DiffusionDet: flat, no hierarchy, no multi-label ----
# INTERPRETATION (ambiguous in the released code, logged as a deviation): the
# vendored head indexes NUM_CLASSES as a 3-list (head.py:81-83), so a genuinely
# single-head DiffusionDet cannot be configured — the repo's own enumeration
# config (NUM_CLASSES: 32, scalar) would raise on construction. The flat label
# space is therefore expressed in the DATA: the target tier's labels move into
# category_id_1 and the other tiers are nulled, so exactly one head is
# supervised. Everything else matches our models, which makes the difference
# against Ours_wo_Manip_Transfer exactly the multi-label head structure.
TIER_CLASSES = {0: 4, 1: 8, 2: 4}
TIER_SPLIT = {0: "quadrant_train", 1: "quadrant_enumeration_train", 2: "diagnosis_train"}
base_records = []
for tier in run.base_tiers:
    flat_train = data_convert.flat_json_path(tier, "train")
    assert os.path.exists(flat_train), "run notebook 01 first: {}".format(flat_train)
    print("\n=== base_diffusiondet_tier{} ===".format(tier))
    record = train_utils.train_stage(
        "base_diffusiondet_tier{}".format(tier), CFG["base_diffusiondet"], run,
        TIER_SPLIT[tier], IMAGENET_WEIGHTS, NUM_GPUS, run.base_diffusiondet,
        extra_overrides=["MODEL.DiffusionDet.NUM_CLASSES",
                         "[{}, 8, 4]".format(TIER_CLASSES[tier])],
        env_override={"TRAIN_JSON": flat_train,
                      "VAL_JSON": data_convert.flat_json_path(tier, "test"),
                      "VAL_IMG_DIR": paths["img_test"], "TIER": "0"})
    record.update({"tier": tier, "flat_train_json": flat_train, "kind": "base_diffusiondet"})
    base_records.append(record)
    training_records.append(record)
if not run.base_tiers:
    print("base DiffusionDet skipped in RUN_MODE={}".format(run.mode))


In [ ]:
# ---- Deviations this notebook establishes ----
setup_env.log_deviation(
    "AMP (SOLVER.AMP.ENABLED=True) enabled for all training",
    "the released configs leave AMP off; T4s need it to fit Swin-B in 16 GB at a "
    "usable throughput, and AMPTrainer is the repo's own code path", "02_train_all",
    impact="mixed-precision reduction order is a residual source of nondeterminism")
setup_env.log_deviation(
    "EMA (MODEL_EMA.ENABLED) left OFF for every run",
    "the repo ships EMA hooks but no config enables them and the paper does not "
    "mention EMA; held constant across variants so it cannot explain any difference",
    "02_train_all")
setup_env.log_deviation(
    "Swin backbone loaded from a .pth, not the config's .pkl filename",
    "DetectionCheckpointer dispatches on file extension: a torch checkpoint named "
    ".pkl is parsed as a Caffe2 blob and the backbone silently stays random",
    "02_train_all",
    impact="fixes a silent failure; without it every number would come from a "
           "randomly initialized backbone")
setup_env.log_deviation(
    "Swin-B used where the released nonpretrain config said SWIN.SIZE: L-22k",
    "that config's own MODEL.WEIGHTS points at a Swin-B checkpoint, so the released "
    "file is internally inconsistent; Swin-B is also what fits a 16 GB T4",
    "02_train_all",
    impact="a Swin-L run would have more capacity; this is a lower bound")
setup_env.log_deviation(
    "SimMIM pretraining on the 1,571 unlabelled X-rays skipped",
    "the authors' SimMIM checkpoint is not published and the pretraining lives in a "
    "separate repository; initialization follows the repo's own nonpretrain config",
    "02_train_all",
    impact="our backbone has never seen a panoramic radiograph; any gap against the "
           "paper is confounded by this")

skipped = [v for v in setup_env.ALL_VARIANTS if v not in run.variants]
if skipped:
    setup_env.log_deviation(
        "diagnosis variants {} not trained".format(skipped),
        "RUN_MODE={} trades variant coverage for matched per-variant compute".format(run.mode),
        "02_train_all",
        impact="those rows carry the original's numbers, clearly marked, and no "
               "reproduced value")
if not run.base_tiers:
    setup_env.log_deviation(
        "base DiffusionDet not trained",
        "RUN_MODE={} spends its budget on the diagnosis ablation, which is the "
        "paper's central claim".format(run.mode), "02_train_all",
        impact="the DiffusionDet_base row carries the original's numbers as "
               "untested context")
print(open(setup_env.DEVIATIONS_MD).read())


In [ ]:
# ---- Notebook summary (the only cross-notebook contract) ----
summary = {
    "run_mode": run.mode,
    "num_gpus": NUM_GPUS,
    "multi_gpu": ddp,
    "preflight": smoke,
    "calibration": calibration,
    "records": training_records,
    "variants_trained": list(run.variants),
    "variants_skipped": skipped,
    "matched_budgets": matched,
    "weights": {
        "imagenet": IMAGENET_WEIGHTS,
        "quadrant": QUADRANT_WEIGHTS,
        "enumeration": ENUM_WEIGHTS,
        "variants": {r["variant"]: train_utils.final_weights(r["name"])
                     for r in variant_records},
        "base": {str(r["tier"]): train_utils.final_weights(r["name"])
                 for r in base_records},
    },
    "noisy_boxes": {"for_enumeration": enum_boxes, "for_diagnosis": diagnosis_boxes,
                    "prior_over_test": prior_test, "stats": box_stats},
    "trajectory_checkpoints": {r["variant"]: r.get("trajectory", {})
                               for r in variant_records},
    "gpu_hours_spent": setup_env.gpu_hours_spent(),
}

path = setup_env.write_notebook_summary("02_train_all", summary)
print("wrote", path)
print(json.dumps(summary, indent=2, default=str)[:4000])


In [ ]:
# ---- Publish, last: the summary above must be inside what gets published ----
# Notebook 03 reads that summary to find the checkpoints, the noisy-box dumps
# and the trajectory snapshot names, so it has to be in the dataset.
publish = {"status": "disabled"}
if PUBLISH_KAGGLE_DATASET:
    publish = train_utils.publish_kaggle_dataset(
        CKPT_DATASET_SLUG, [setup_env.RUNS_DIR, setup_env.PAPER_ASSETS],
        "training ({} mode)".format(run.mode))
    summary["kaggle_publish"] = {k: v for k, v in publish.items()
                                 if k not in ("stdout", "stderr")}
    setup_env.write_notebook_summary("02_train_all", summary)
print(json.dumps({k: v for k, v in publish.items() if k not in ("stdout", "stderr")},
                 indent=2))
print("\nGPU-hours recorded so far: {:.2f}".format(setup_env.gpu_hours_spent()))
print("Attach to the next session as: {} (plus {})".format(
    CKPT_DATASET_SLUG, DATA_DATASET_SLUG))
